# 第三部分：描述性统计、可视化与 CAPM 回归

本 Notebook 完成以下分析：
1. 日收益率描述性统计（含最大回撤）
2. 归一化收盘价走势图（图 1）
3. 收益率分面直方图（图 2）
4. 收益率相关性热力图（图 3）
5. 宏观指标与股市关系散点图（图 4）
6. CAPM 回归与 Beta 系数点图（图 5）
7. 财务指标跨公司对比（图 6，选做）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import warnings, os, time
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150

os.makedirs("output", exist_ok=True)

df = pd.read_csv("data/combined/combined_data.csv", parse_dates=["date"])
print(f"数据维度: {df.shape}")
print(f"日期范围: {df['date'].min().date()} 至 {df['date'].max().date()}")
print(f"股票数量: {df['code'].nunique()}")

stocks = [
    {"code": "000001", "name": "平安银行", "industry": "银行"},
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪",   "industry": "汽车"},
    {"code": "300750", "name": "宁德时代", "industry": "能源"},
    {"code": "601012", "name": "隆基绿能", "industry": "能源"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
    {"code": "600048", "name": "保利发展", "industry": "房地产"},
    {"code": "002475", "name": "立讯精密", "industry": "通讯"},
]
stocks_df = pd.DataFrame(stocks)

RF_ANNUAL = 0.02
RF_DAILY = RF_ANNUAL / 252

## 4.1 描述性统计

计算日对数收益率 $r_t = \ln(P_t / P_{t-1})$ 的描述性统计：
年化均值、年化波动率、偏度、峰度、最大回撤。

In [ ]:
def max_drawdown(series):
    """计算最大回撤"""
    cum = (1 + series).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min()

stats_list = []
for stock in stocks:
    code_ = stock["code"]
    s = df[df["code"] == code_]["log_return"].dropna()
    ann_mean = s.mean() * 252
    ann_vol = s.std() * np.sqrt(252)
    stats_list.append({
        "股票": stock["name"],
        "行业": stock["industry"],
        "年化均值": round(ann_mean, 4),
        "年化波动率": round(ann_vol, 4),
        "偏度": round(s.skew(), 4),
        "峰度": round(s.kurtosis(), 4),
        "最大回撤": round(max_drawdown(s), 4),
    })

stats_df = pd.DataFrame(stats_list)
print("=== 日对数收益率描述性统计 ===")
display(stats_df)
stats_df.to_csv("output/descriptive_stats.csv", index=False, encoding="utf-8-sig")

## 4.2 图 1：归一化收盘价走势图

以 **2020 年初 = 1** 为基准，叠加沪深 300，按行业分组着色。

In [ ]:
base_date = pd.Timestamp("2020-01-02")

industry_colors = {
    "银行": "#2196F3", "汽车": "#FF5722", "能源": "#4CAF50",
    "白酒": "#9C27B0", "通讯": "#FF9800", "物流": "#795548",
    "房地产": "#607D8B"
}

fig, ax = plt.subplots(figsize=(14, 7))

# 沪深 300 归一化
hs300 = df[["date", "idx_close"]].dropna().drop_duplicates("date")
hs300 = hs300[hs300["date"] >= base_date].sort_values("date")
base_val = hs300.iloc[0]["idx_close"]
ax.plot(hs300["date"], hs300["idx_close"] / base_val,
        "k--", linewidth=1.5, alpha=0.7, label="沪深300")

for stock in stocks:
    code_ = stock["code"]
    name_ = stock["name"]
    ind = stock["industry"]
    s = df[df["code"] == code_][["date", "close"]].dropna()
    s = s[s["date"] >= base_date].sort_values("date")
    if len(s) == 0:
        continue
    bval = s.iloc[0]["close"]
    ax.plot(s["date"], s["close"] / bval,
            color=industry_colors.get(ind, "gray"),
            linewidth=1.2, label=f"{name_}({ind})")

ax.axhline(y=1, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("日期", fontsize=12)
ax.set_ylabel("归一化价格（2020-01-02 = 1）", fontsize=12)
ax.set_title("图 1：归一化收盘价走势（2020-01-02 = 1，按行业着色）",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper left", fontsize=8, ncol=3)
plt.tight_layout()
plt.savefig("output/fig1_normalized_price.png", dpi=150, bbox_inches="tight")
plt.show()
print("图 1 已保存。")

**解读**：以 2020 年初为基准（=1），可以直观比较各股票的累计涨跌表现。行业颜色分组显示：新能源（宁德时代、隆基绿能）和汽车（比亚迪）涨幅显著，银行股整体表现平稳，房地产（保利发展）自 2021 年后持续承压。

## 4.3 图 2：日收益率分布直方图

10 只股票收益率分面直方图（2 行 x 5 列），每个子图叠加正态分布曲线，标注均值和标准差。

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, stock in enumerate(stocks):
    code_ = stock["code"]
    name_ = stock["name"]
    ind = stock["industry"]
    s = df[df["code"] == code_]["log_return"].dropna()
    mu, sigma = s.mean(), s.std()

    axes[idx].hist(s, bins=50, density=True, alpha=0.6,
                    color=industry_colors.get(ind, "gray"), edgecolor="white")
    x = np.linspace(s.min(), s.max(), 200)
    axes[idx].plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=1.5)
    axes[idx].axvline(mu, color="blue", linestyle="--", linewidth=1)
    axes[idx].set_title(f"{name_}\n$\mu$={mu:.4f}, $\sigma$={sigma:.4f}", fontsize=9)
    axes[idx].set_xlabel("日对数收益率")
    if idx % 5 == 0:
        axes[idx].set_ylabel("密度")

plt.suptitle("图 2：日对数收益率分布直方图（叠加正态拟合）",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("output/fig2_return_histogram.png", dpi=150, bbox_inches="tight")
plt.show()
print("图 2 已保存。")

**解读**：所有股票收益率分布均呈现"尖峰厚尾"特征（峰度 > 0），偏离正态分布假设。极端事件（大涨大跌）发生的频率远高于正态分布的预测，这与金融学中的经验发现一致。

## 4.4 图 3：收益率相关性热力图

10 只股票日收益率的相关系数矩阵，**按行业排序**，标注数值。

In [ ]:
industry_order = ["银行", "汽车", "能源", "白酒", "通讯", "物流", "房地产"]
codes_by_ind = {}
for stock in stocks:
    codes_by_ind.setdefault(stock["industry"], []).append(stock["code"])

ordered_codes = []
for ind in industry_order:
    ordered_codes.extend(codes_by_ind.get(ind, []))

return_wide = df.pivot_table(
    index="date", columns="code", values="log_return", aggfunc="first"
)
return_wide = return_wide.reindex(columns=ordered_codes)

name_map = {s["code"]: s["name"] for s in stocks}
return_wide.columns = [name_map[c] for c in return_wide.columns]
corr = return_wide.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("图 3：日收益率相关系数热力图（按行业排序）",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output/fig3_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("图 3 已保存。")

**解读**：同行业股票相关性通常高于跨行业。两只银行股（平安银行 vs 招商银行）相关系数较高，体现了同行业的系统性影响。新能源板块（宁德时代 vs 隆基绿能）相关性也高于与其他行业的配对，说明行业因素是驱动股票收益联动的重要来源。

## 4.5 图 4：宏观指标与股市关系

绘制 CPI 同比增速与沪深 300 月度收益率的散点图，叠加线性拟合线，标注 Pearson 相关系数。

In [ ]:
# 计算沪深 300 月度对数收益率
hs300_monthly = df[["date", "idx_close"]].drop_duplicates("date").copy()
hs300_monthly = hs300_monthly.set_index("date").resample("ME").last()
hs300_monthly["idx_monthly_return"] = np.log(
    hs300_monthly["idx_close"] / hs300_monthly["idx_close"].shift(1)
)
hs300_monthly = hs300_monthly.dropna().reset_index()
hs300_monthly["year_month"] = hs300_monthly["date"].dt.to_period("M")

# 读取 CPI
cpi_df = pd.read_csv("data/macro/macro_cpi.csv")
date_col = "日期"
val_cols = [c for c in cpi_df.columns if c in ["今值", "cpi_yoy"]]
cpi_df["date"] = pd.to_datetime(cpi_df[date_col], errors="coerce")
cpi_df["cpi_yoy"] = pd.to_numeric(cpi_df[val_cols[0]], errors="coerce") if val_cols else np.nan
cpi_df["year_month"] = cpi_df["date"].dt.to_period("M")

# 合并
scatter_df = pd.merge(
    hs300_monthly[["year_month", "idx_monthly_return"]],
    cpi_df[["year_month", "cpi_yoy"]], on="year_month", how="inner"
).dropna()

r, p_val = stats.pearsonr(scatter_df["cpi_yoy"], scatter_df["idx_monthly_return"])

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(scatter_df["cpi_yoy"], scatter_df["idx_monthly_return"],
           alpha=0.6, s=40, color="steelblue")

coef = np.polyfit(scatter_df["cpi_yoy"], scatter_df["idx_monthly_return"], 1)
x_fit = np.linspace(scatter_df["cpi_yoy"].min(), scatter_df["cpi_yoy"].max(), 100)
y_fit = np.polyval(coef, x_fit)
ax.plot(x_fit, y_fit, "r-", linewidth=2, label="拟合线")

ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("CPI 同比增速 (%)", fontsize=12)
ax.set_ylabel("沪深 300 月度对数收益率", fontsize=12)
ax.set_title(
    f"图 4：CPI 同比增速 vs 沪深 300 月度收益率\n"
    f"Pearson r = {r:.3f} (p = {p_val:.4f})",
    fontsize=13, fontweight="bold"
)
ax.legend()
plt.tight_layout()
plt.savefig("output/fig4_macro_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("图 4 已保存。")

**解读**：CPI 同比增速与沪深 300 月度收益率的相关系数为 $r$（见上图标题）。正相关意味着通胀上升时股市可能受益于名义盈利增长，负相关则可能因加息预期承压。需注意相关关系不等于因果关系。

## 5.1 CAPM 模型估计

对 10 只股票分别估计 CAPM 模型：

$$r_{i,t} - r_f = \alpha_i + \beta_i (r_{m,t} - r_f) + \varepsilon_{i,t}$$

- $r_{i,t}$：个股日对数收益率
- $r_{m,t}$：沪深 300 日对数收益率  
- $r_f^{daily} = 0.02 / 252$

In [ ]:
capm_results = []

for stock in stocks:
    code_ = stock["code"]
    name_ = stock["name"]
    ind = stock["industry"]
    s = df[df["code"] == code_][["date", "log_return", "idx_log_return"]].dropna()

    y = s["log_return"].values - RF_DAILY
    X = sm.add_constant(s["idx_log_return"].values - RF_DAILY)

    model = sm.OLS(y, X).fit()
    ci_low, ci_high = model.conf_int()[1]

    capm_results.append({
        "股票": name_, "行业": ind,
        "alpha": round(model.params[0], 6),
        "alpha_p": round(model.pvalues[0], 4),
        "beta": round(model.params[1], 4),
        "beta_p": round(model.pvalues[1], 4),
        "beta_CI_low": round(ci_low, 4),
        "beta_CI_high": round(ci_high, 4),
        "R2": round(model.rsquared, 4)
    })

capm_df = pd.DataFrame(capm_results)
print("=== CAPM 回归结果 ===")
display(capm_df)
capm_df.to_csv("output/capm_results.csv", index=False, encoding="utf-8-sig")

### Beta 系数点图

横轴为 Beta 值，纵轴为股票名称，误差棒表示 95% 置信区间，按行业分组着色，$\beta=1$ 参考竖线。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

capm_df_sorted = capm_df.sort_values("beta").reset_index(drop=True)
y_pos = range(len(capm_df_sorted))

for i, row in capm_df_sorted.iterrows():
    color = industry_colors.get(row["行业"], "gray")
    err_lo = row["beta"] - row["beta_CI_low"]
    err_hi = row["beta_CI_high"] - row["beta"]
    ax.errorbar(row["beta"], i,
                xerr=[[err_lo], [err_hi]],
                fmt="o", color=color, capsize=4, markersize=8, elinewidth=2)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(capm_df_sorted["股票"])
ax.axvline(x=1, color="black", linestyle="--", linewidth=1.5, label="$\beta$=1")
ax.set_xlabel("Beta 系数", fontsize=12)
ax.set_title("图 5：CAPM Beta 系数点图（95% 置信区间，按行业着色）",
             fontsize=13, fontweight="bold")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=industry_colors.get(ind, "gray"), label=ind)
    for ind in industry_order
]
legend_elements.append(plt.Line2D([0], [0], color="black", linestyle="--", label="$\beta$=1"))
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("output/fig5_beta_dotplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("图 5 已保存。")

### CAPM 回归讨论

**讨论问题 1：哪些股票 $\beta > 1$？它们属于哪些行业？这与"周期性 vs 防御性"行业分类是否吻合？**

In [ ]:
aggressive = capm_df[capm_df["beta"] > 1]
defensive = capm_df[capm_df["beta"] <= 1]

print("Beta > 1 的股票（进攻型）:")
for _, r in aggressive.iterrows():
    print(f"  {r['股票']}({r['行业']}): beta = {r['beta']:.3f}")

print("\nBeta <= 1 的股票（防御型）:")
for _, r in defensive.iterrows():
    print(f"  {r['股票']}({r['行业']}): beta = {r['beta']:.3f}")

**分析**：$\beta > 1$ 的股票波动大于市场，属于"进攻型"，通常集中在周期性行业。$\beta \leq 1$ 的股票波动小于市场，属于"防御型"。银行股的 $\beta$ 通常较低，因为银行业盈利受经济周期影响相对间接，且银行股股息率较高提供了一定"安全垫"。新能源和汽车行业 $\beta$ 较高，反映了对经济景气度和政策变化的敏感性。

**讨论问题 2：$\alpha$ 是否显著异于零？Alpha 显著意味着什么？**

In [ ]:
for _, r in capm_df.iterrows():
    sig = "显著" if r["alpha_p"] < 0.05 else "不显著"
    print(f"  {r['股票']}: alpha = {r['alpha']:.6f}, p = {r['alpha_p']:.4f} -> {sig}")

**分析**：在 CAPM 框架下，$\alpha$ 代表"超额收益"，即无法被市场风险（$\beta$）解释的收益部分。如果 $\alpha$ 显著为正，说明该股票在控制市场风险后仍有超额回报，可能来源于选股能力、信息优势或市场有效性不足。如果 $\alpha$ 显著为负，说明该股票表现劣于 CAPM 预测。如果 $\alpha$ 不显著，说明 CAPM 模型能较好地解释该股票的收益。

**讨论问题 3：$R^2$ 最高和最低的股票分别是哪只？如何解释这一差异？**

In [ ]:
max_r2 = capm_df.loc[capm_df["R2"].idxmax()]
min_r2 = capm_df.loc[capm_df["R2"].idxmin()]
print(f"R2 最高: {max_r2['股票']}({max_r2['行业']}), R2 = {max_r2['R2']:.4f}")
print(f"R2 最低: {min_r2['股票']}({min_r2['行业']}), R2 = {min_r2['R2']:.4f}")

**分析**：$R^2$ 反映市场因子对个股收益的解释力。$R^2$ 越高，说明该股票收益变动越能被市场整体走势解释；$R^2$ 越低，说明个股特质因素（如公司基本面、行业事件）对收益影响更大。银行股通常 $R^2$ 较高，因为银行板块与宏观经济高度联动；而某些行业龙头（如茅台）可能因独特的基本面因素而 $R^2$ 较低。

## 图 6（选做）：财务指标跨公司对比

绘制 10 只股票近 5 年 ROE 的折线图，按行业分组。

In [ ]:
try:
    fin = pd.read_csv("data/finance/finance_ratios.csv")
    roe = fin[fin["indicator"] == "ROE"].copy()
    if len(roe) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        for stock in stocks:
            s = roe[roe["code"] == stock["code"]]
            if len(s) > 0:
                ax.plot(s["year"], s["value"], "o-",
                        color=industry_colors.get(stock["industry"], "gray"),
                        label=f"{stock['name']}({stock['industry']})",
                        linewidth=1.5, markersize=5)
        ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
        ax.set_xlabel("年份", fontsize=12)
        ax.set_ylabel("ROE (%)", fontsize=12)
        ax.set_title("图 6：各股票 ROE 趋势对比（按行业着色）",
                     fontsize=13, fontweight="bold")
        ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        plt.savefig("output/fig6_roe_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("图 6 已保存。")
    else:
        print("ROE 数据为空，跳过图 6。")
except Exception as e:
    print(f"图 6 跳过: {e}")

## 总结

In [ ]:
print("分析完成，输出文件清单：")
for f in sorted(os.listdir("output")):
    size = os.path.getsize(f"output/{f}") / 1024
    print(f"  output/{f}: {size:.1f} KB")